In [27]:
import os
import requests
import csv
from datetime import date, datetime, timedelta
import pandas as pd

# match_redcap_termination.py

REDCAP_API_TOKEN = 'B40BBD579D769085375A5179F942F093'
REDCAP_API_URL = "https://population.ahri.org/api/"

# ============================================================
# CONFIG — update these
# ============================================================

CSV_INPUT_PATH = r"C:/Users/Jabulani.Mcineka/workspace/PySpark-Essential-Training/data/output1.csv"
CSV_MATCH_COLUMN = "record"
CSV_OUTPUT_PATH = r"C:/Users/Jabulani.Mcineka/workspace/PySpark-Essential-Training/data/matched_output.csv"
REDCAP_EXPORT_PATH = r"C:/Users/Jabulani.Mcineka/workspace/PySpark-Essential-Training/data/redcap_export.csv"


# ============================================================
# Read data from REDCap
# ============================================================
def get_redcap_termination_data(token=REDCAP_API_TOKEN, url=REDCAP_API_URL):
    payload = {
        'token': token,
        'content': 'record',
        'action': 'export',
        'format': 'json',
        'type': 'flat',
        'fields[0]': 'id_record',
        'fields[1]': 'termination_date',
        'fields[2]': 'dod',
        'events[0]': 'termination_arm_1',
        'rawOrLabel': 'raw',
        'rawOrLabelHeaders': 'raw',
        'exportCheckboxLabel': 'false',
        'exportSurveyFields': 'false',
        'exportDataAccessGroups': 'false',
        'returnFormat': 'json'
    }

    try:
        response = requests.post(url, data=payload, timeout=30)
        response.raise_for_status()
        records = response.json()
    except Exception as exc:
        print("Unable to fetch REDCap data:", exc)
        return {}

    lookup = {}
    for record in records:
        record_id = str(record.get('id_record', '')).strip()
        if record_id:
            lookup[record_id] = {
                'termination_date': record.get('termination_date', ''),
                'dod': record.get('dod', '')
            }

    return lookup


# ============================================================
# MAIN
# ============================================================

redcap_lookup = get_redcap_termination_data()

print(f"Retrieved {len(redcap_lookup)} records")

if redcap_lookup:
    export_df = pd.DataFrame.from_dict(redcap_lookup, orient='index').reset_index().rename(columns={'index': 'record_id'})
    export_df.to_csv(REDCAP_EXPORT_PATH, index=False)
    print(f"Saved {len(export_df)} records to {REDCAP_EXPORT_PATH}")
    display(export_df.head())
else:
    print('No REDCap data was returned.')

Retrieved 583 records
Saved 583 records to C:/Users/Jabulani.Mcineka/workspace/PySpark-Essential-Training/data/redcap_export.csv


,record_id,termination_date,dod
0,039-02-0001,2023-09-04,
1,039-02-0002,2024-02-14,
2,039-02-0003,2023-07-31,
3,039-02-0004,2023-07-31,
4,039-02-0005,2023-07-31,
